# Step 3: Placebo Test, Features and Baseline Models

**This step answers two questions**
1. Can the labels be trusted? A **placebo test** measures the false-positive rate
2. Can pre-event information predict "outage longer than 7 days"? Does the model still work on a typhoon it has never seen? This is the core result for RQ2

| Part | Content | Needs Earth Engine? |
| --- | --- | --- |
| A | Placebo test: pretend each typhoon made landfall 1–2 lunar months earlier (same moon phase) and run the same detection pipeline | No |
| B | Features: wind speed (parametric wind field), rainfall, population, built-up area, elevation, distance to coast, pre-event light, AlphaEarth satellite embeddings | Yes (exports take about 10–30 minutes) |
| C | Models: logistic regression and gradient boosting, three feature sets × three validation schemes | No |
| D | Diagnostics: spatial autocorrelation of residuals (Moran's I), and performance by pre-event light and relative wealth | No |

**Inputs**: the Step 2/2b files in Drive (`step2b_labels.csv`, `ntl_*.csv`, `meta_*.csv`)

**Optional (for the equity analysis in Part D)**: Meta's Relative Wealth Index (RWI). Download the Philippines CSV from the HDX [Relative Wealth Index](https://data.humdata.org/dataset/relative-wealth-index) page and upload it to the `ph_typhoon_ntl` folder in Drive; any file name containing `relative_wealth` works. Without it the notebook still runs and simply skips the RWI breakdown.

**Before running**: replace `PROJECT_ID` in cell 2 with your own project ID.

In [ ]:
# 1. Install dependencies
!pip install -q earthengine-api geemap geopandas scikit-learn

In [ ]:
# 2. Authenticate Earth Engine and mount Drive
import os, glob, time
import ee
PROJECT_ID = "ee-yourname"   # replace with your own Cloud project ID
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

from google.colab import drive
drive.mount("/content/drive")
DRIVE_FOLDER = "ph_typhoon_ntl"
DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs("figures", exist_ok=True)
print("OK")

In [ ]:
# 3. Events and parameters; label functions (as in Step 2b, with bright-moon nights removed from the nightly analysis too)
import numpy as np
import pandas as pd

EVENTS = pd.DataFrame([
    ("haiyan_2013",   "HAIYAN",   2013, "Yolanda", "2013-11-08 04:40", None,         "reference"),
    ("mangkhut_2018", "MANGKHUT", 2018, "Ompong",  "2018-09-15 01:40", None,         "model"),
    ("kammuri_2019",  "KAMMURI",  2019, "Tisoy",   "2019-12-02 23:00", None,         "model"),
    ("phanfone_2019", "PHANFONE", 2019, "Ursula",  "2019-12-24 16:45", "2019-11-28", "model"),
    ("rai_2021",      "RAI",      2021, "Odette",  "2021-12-16 13:30", None,         "model"),
], columns=["key", "ibtracs_name", "season", "local_name", "landfall", "baseline_end", "role"])
EVENTS["landfall"] = pd.to_datetime(EVENTS["landfall"])
EV = EVENTS.set_index("key")
MODEL_KEYS = EVENTS.loc[EVENTS.role == "model", "key"].tolist()

BASE_DAYS, BASE_GAP_DAYS, POST_WEEKS = 120, 3, 26
MIN_ELIGIBLE_PX, MIN_VALID_FRAC, MOON_MAX_ILLUM = 8, 0.5, 0.8
MIN_DAYS_PER_WEEK, MIN_BASE_NIGHTS = 2, 8
OUTAGE_LEVEL = RECOVERY_LEVEL = 80
DAILY_DAYS, ROLL_N, MAX_SPAN_DAYS, FIRST_LOOK_DAYS = 28, 3, 6, 5
OVERPASS_LAG = pd.Timedelta(hours=25, minutes=30)
REF_NEW_MOON, SYNODIC = pd.Timestamp("2000-01-06 18:14"), 29.530588853
LONG_OUTAGE_DAYS = 7          # prediction target: outage longer than 7 days
BUILT_MIN_FRAC = 0.03

def windows(ev):
    base_end = pd.Timestamp(ev.baseline_end) if isinstance(ev.baseline_end, str) else ev.landfall - pd.Timedelta(days=BASE_GAP_DAYS)
    return base_end - pd.Timedelta(days=BASE_DAYS), base_end

def moon_illum(obs_utc):
    age = ((obs_utc - REF_NEW_MOON) / pd.Timedelta(days=1)) % SYNODIC
    return (1 - np.cos(2 * np.pi * age / SYNODIC)) / 2

def load_event(key):
    files = [f for f in sorted(glob.glob(f"{DRIVE_DIR}/ntl_{key}*.csv")) if "_px4" not in os.path.basename(f)]
    if not files:
        return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    d["date"] = pd.to_datetime(d["date"])
    d["obs_pht"] = d["date"] + OVERPASS_LAG
    d["illum"] = moon_illum(d["obs_pht"] - pd.Timedelta(hours=8))
    return d

def week_of(obs, t0):
    return (np.ceil((obs - t0) / pd.Timedelta(days=1) / 7) - 1).astype(int)

def label_all(g, t0, base_start, base_end, censor_at):
    ok = g["mean"].notna() & (g["count"] >= MIN_VALID_FRAC * g["eligible_px"]) & (g["illum"] <= MOON_MAX_ILLUM)
    g = g[ok].sort_values("obs_pht")
    base = g[(g.obs_pht >= base_start) & (g.obs_pht <= base_end)]
    out = {"base_nights": len(base)}
    if len(base) < 3:
        return {**out, "status_w": "no_baseline", "status_d": "no_baseline"}
    b = base["mean"].median()
    out["baseline_radiance"] = b
    g = g.assign(pct=g["mean"] / b * 100, wk=week_of(g.obs_pht, t0))
    end = t0 + pd.Timedelta(weeks=POST_WEEKS)
    if pd.notna(censor_at):
        end = min(end, censor_at)
    pre = g.loc[base.index].groupby("wk").pct.agg(["median", "size"])
    noise = pre.loc[pre["size"] >= MIN_DAYS_PER_WEEK, "median"].std()
    w = g[(g.obs_pht > t0) & (g.obs_pht < end)].groupby("wk").pct.agg(["median", "size"])
    w = w.loc[w["size"] >= MIN_DAYS_PER_WEEK, "median"]
    early = w[w.index <= 3]
    if early.empty:
        out["status_w"] = "no_early_data"
    else:
        low, low_wk = early.min(), early.idxmin()
        thr = OUTAGE_LEVEL if pd.isna(noise) else min(OUTAGE_LEVEL, 100 - 2 * noise)
        out.update(outage_w=bool(low < thr), status_w="ok" if len(base) >= MIN_BASE_NIGHTS else "low_baseline")
        if not out["outage_w"]:
            out.update(recovery_weeks=0, censored_w=False)
        else:
            after = w[w.index >= low_wk]
            hit = [k for k, a, c in zip(after.index[:-1], after.values[:-1], after.values[1:])
                   if a >= RECOVERY_LEVEL and c >= RECOVERY_LEVEL]
            out.update(recovery_weeks=int(hit[0]) + 1 if hit else int(w.index.max()) + 1, censored_w=not hit)
    bp = g.loc[base.index, "pct"].values
    rolls_b = [np.median(bp[i:i + ROLL_N]) for i in range(len(bp) - ROLL_N + 1)]
    noise3 = np.std(rolls_b, ddof=1) if len(rolls_b) > 2 else np.nan
    thr_d = OUTAGE_LEVEL if pd.isna(noise3) else min(OUTAGE_LEVEL, 100 - 2 * noise3)
    post = g[(g.obs_pht > t0) & (g.obs_pht < min(end, t0 + pd.Timedelta(days=DAILY_DAYS)))]
    days = ((post.obs_pht - t0) / pd.Timedelta(days=1)).values
    vals = post.pct.values
    rolls = [(days[i], np.median(vals[i:i + ROLL_N])) for i in range(len(vals) - ROLL_N + 1)
             if days[i + ROLL_N - 1] - days[i] <= MAX_SPAN_DAYS]
    if not rolls:
        out["status_d"] = "too_few_nights"
    elif rolls[0][0] > FIRST_LOOK_DAYS:
        out["status_d"] = "late_first_look"
    else:
        rd, rv = np.array([r[0] for r in rolls]), np.array([r[1] for r in rolls])
        ei = np.where(rd <= FIRST_LOOK_DAYS)[0]
        i_min = ei[np.argmin(rv[ei])]
        out.update(outage_d=bool(rv[i_min] < thr_d), status_d="ok")
        if not out["outage_d"]:
            out.update(outage_days=0.0, censored_d=False)
        else:
            rec = np.where((rv >= RECOVERY_LEVEL) & (np.arange(len(rv)) > i_min))[0]
            out.update(outage_days=float(rd[rec[0]]) if len(rec) else float(days[-1]), censored_d=not len(rec))
    return out

def combine(r):
    wk_ok = r.get("status_w") in ("ok", "low_baseline")
    d_ok = r.get("status_d") == "ok"
    wk_out, d_out = wk_ok and bool(r.get("outage_w")), d_ok and bool(r.get("outage_d"))
    if d_out and not r["censored_d"]:
        return r["outage_days"], False
    if d_out:
        return (max(DAILY_DAYS, r["recovery_weeks"] * 7), bool(r["censored_w"])) if wk_out else (r["outage_days"], True)
    if wk_out:
        return r["recovery_weeks"] * 7, bool(r["censored_w"])
    return (0.0, False) if (d_ok or wk_ok) else (np.nan, np.nan)

In [ ]:
# 4. Load Step 2b labels, typhoon tracks and municipal boundaries
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

labels = pd.read_csv(f"{DRIVE_DIR}/step2b_labels.csv", encoding="utf-8-sig")
model_rows = labels[labels.use_for_model.astype(str).str.lower() == "true"].copy()
print(model_rows.groupby("event").size().rename("municipalities in modelling sample"))

IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
               "v04r01/access/csv/ibtracs.WP.list.v04r01.csv")
cols = ["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "USA_WIND", "WMO_WIND", "USA_RMW"]
ib = pd.read_csv(IBTRACS_URL, usecols=cols, skiprows=[1], low_memory=False)
ib["SEASON"] = pd.to_numeric(ib["SEASON"], errors="coerce")
ib = ib[ib["SEASON"] >= 2013].copy()
for c in ["LAT", "LON", "USA_WIND", "WMO_WIND", "USA_RMW"]:
    ib[c] = pd.to_numeric(ib[c], errors="coerce")
ib["time_pht"] = pd.to_datetime(ib["ISO_TIME"]) + pd.Timedelta(hours=8)
tracks = {}
for ev in EVENTS.itertuples():
    t = ib[(ib.NAME == ev.ibtracs_name) & (ib.SEASON == ev.season)]
    if t.SID.nunique() > 1:
        t = t[t.SID == t.assign(g=(t.time_pht - ev.landfall).abs()).groupby("SID").g.min().idxmin()]
    tracks[ev.key] = t.sort_values("time_pht")

def load_gb(level):
    try:
        meta = requests.get(f"https://www.geoboundaries.org/api/current/gbOpen/PHL/{level}/", timeout=60).json()
        return gpd.read_file(meta["simplifiedGeometryGeoJSON"])
    except Exception as e:
        print(f"{level}: API failed, trying the GitHub mirror ({e})")
        return gpd.read_file("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/main/releaseData/"
                             f"gbOpen/PHL/{level}/geoBoundaries-PHL-{level}_simplified.geojson")

adm3 = load_gb("ADM3").to_crs(4326)[["shapeID", "geometry"]]
UTM = 32651
adm3_utm = adm3.to_crs(UTM)
anchor = adm3_utm.set_geometry(adm3_utm.representative_point())
anchor_ll = anchor.to_crs(4326)

INK, INK2, SURFACE, GRID = "#1f1f1e", "#5f5e5a", "#fcfcfb", "#e5e4e0"
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.edgecolor": "#c9c8c4", "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE})

## A. Placebo test

For each typhoon, shift the "landfall" back by a whole number of lunar months (identical moon phase), making sure the 4 weeks after the fake landfall fall inside the real, storm-free baseline period. Then detect outages with **exactly the same pipeline**.

- Share of municipalities flagged at the fake landfall = **false-positive rate**
- The real outage share should be far above it; the bigger the gap, the more trustworthy the labels
- The false-positive rate should not depend on distance (a fake typhoon has no track)

The same cell also computes each municipality's pre-event radiance, used as a feature in Part B.

In [ ]:
# 5. Placebo test + pre-event radiance
placebo_rows, pre_ntl_rows, placebo_info = [], [], []
for ev in EVENTS.itertuples():
    d = load_event(ev.key)
    if d is None:
        print(f"{ev.key}: missing export file, skipped")
        continue
    meta = pd.read_csv(f"{DRIVE_DIR}/meta_{ev.key}.csv")
    d = d.merge(meta[["shapeID", "eligible_px"]], on="shapeID", how="left")
    base_start, base_end = windows(ev)
    k = 1
    while ev.landfall - pd.Timedelta(days=k * SYNODIC) + pd.Timedelta(days=28) > base_end:
        k += 1
    t0p = ev.landfall - pd.Timedelta(days=k * SYNODIC)
    pb_end = t0p - pd.Timedelta(days=BASE_GAP_DAYS)
    if (pb_end - base_start).days < 30:
        print(f"{ev.key}: placebo baseline shorter than 30 days, skipped")
        continue
    placebo_info.append({"event": ev.key, "lunar months earlier": k, "placebo landfall": t0p.strftime("%Y-%m-%d %H:%M"),
                         "placebo baseline": f"{base_start:%Y-%m-%d} → {pb_end:%Y-%m-%d}"})
    info = meta.set_index("shapeID")
    for sid, g in d.groupby("shapeID"):
        rp = label_all(g, t0p, base_start, pb_end, base_end)
        dur_p, _ = combine(rp)
        rr = label_all(g, ev.landfall, base_start, base_end, pd.NaT)
        placebo_rows.append({"event": ev.key, "shapeID": sid, "dist_km": info.loc[sid, "dist_km"],
                             "placebo_days": dur_p})
        pre_ntl_rows.append({"event": ev.key, "shapeID": sid, "pre_ntl": rr.get("baseline_radiance", np.nan)})

placebo = pd.DataFrame(placebo_rows)
pre_ntl = pd.DataFrame(pre_ntl_rows)
display(pd.DataFrame(placebo_info))

real = labels[labels.duration_source != "none"][["event", "shapeID", "duration_days", "duration_censored"]]
P = placebo.merge(real, on=["event", "shapeID"], how="inner").dropna(subset=["placebo_days", "duration_days"])
PT = P.groupby("event", sort=False).apply(lambda x: pd.Series({
    "municipalities": len(x),
    "real: any outage": (x.duration_days > 0).mean(),
    "placebo: any outage": (x.placebo_days > 0).mean(),
    f"real: > {LONG_OUTAGE_DAYS} days": (x.duration_days > LONG_OUTAGE_DAYS).mean(),
    f"placebo: > {LONG_OUTAGE_DAYS} days": (x.placebo_days > LONG_OUTAGE_DAYS).mean(),
}), include_groups=False)
PT.round(3)

In [ ]:
# 6. Figure: real vs placebo (share with outage > 7 days, by distance band)
BANDS = [0, 30, 60, 100, 150]
xl = [f"{a}–{b}" for a, b in zip(BANDS[:-1], BANDS[1:])]
evs = PT.index.tolist()
fig, axes = plt.subplots(1, len(evs), figsize=(3.6 * len(evs), 3.4), sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, e in zip(axes, evs):
    x = P[P.event == e]
    band = pd.cut(x.dist_km, BANDS)
    r = (x.duration_days > LONG_OUTAGE_DAYS).groupby(band, observed=False).mean() * 100
    p = (x.placebo_days > LONG_OUTAGE_DAYS).groupby(band, observed=False).mean() * 100
    pos = np.arange(len(xl))
    ax.bar(pos - 0.2, r.values, width=0.38, color=C1, label="Real landfall")
    ax.bar(pos + 0.2, p.values, width=0.38, color="#a3a29d", label="Placebo landfall")
    ax.set_xticks(pos, xl, fontsize=8)
    ax.set_title(f"{e} (n = {len(x)})", loc="left", fontsize=10)
    ax.set_ylim(0, 100)
    ax.grid(axis="y", color=GRID, lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_ylabel(f"Share with outage > {LONG_OUTAGE_DAYS} days (%)")
axes[-1].legend(frameon=False, fontsize=9)
fig.supxlabel("Distance from municipality to track (km)", color=INK2)
fig.suptitle("Placebo test: the same pipeline run at a fake landfall one or two lunar months earlier",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step3_placebo.png", dpi=200, bbox_inches="tight")
plt.show()

## B. Features

**Computed locally** (a few seconds):
- **Maximum wind**: from the IBTrACS track (maximum wind, radius of maximum wind) and a modified Rankine vortex, the maximum sustained wind each municipality experienced, plus hours at or above 17 m/s (gale force). This is a simplified parametric wind field: it ignores the asymmetry from forward motion and terrain effects
- Distance to the track, distance to the coast, municipality area
- Pre-event radiance (already computed in Part A)

**Earth Engine export** (one task per typhoon):
- Rainfall accumulated from 1 day before to 2 days after landfall (GPM IMERG V07)
- Population density (GHSL 2020), built-up fraction (GHSL 2020), mean elevation (SRTM)
- AlphaEarth satellite embeddings from **the year before landfall**, averaged over built-up pixels only (64 dimensions)

In [ ]:
# 7. Local features: parametric wind field, distances, area
KT2MS, NMI2KM, R_EARTH = 0.514444, 1.852, 6371.0

def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1)[:, None], np.radians(lat2)[None, :]
    dphi, dl = p2 - p1, np.radians(lon2)[None, :] - np.radians(lon1)[:, None]
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))

def wind_features(track, landfall, lat, lon):
    t = track[(track.time_pht - landfall).abs() <= pd.Timedelta(days=3)].copy()
    t["vmax"] = (t.USA_WIND * KT2MS).fillna(t.WMO_WIND / 0.88 * KT2MS)   # convert 10-min to 1-min wind
    t["rmw"] = (t.USA_RMW * NMI2KM)
    t = (t.drop_duplicates("time_pht").set_index("time_pht")[["LAT", "LON", "vmax", "rmw"]]
           .resample("30min").mean().interpolate(limit_direction="both"))
    t["rmw"] = t["rmw"].fillna(40).clip(10, 100)
    r = haversine_km(lat, lon, t.LAT.values, t.LON.values)
    R, V = t.rmw.values[None, :], t.vmax.values[None, :]
    v = np.where(r < R, V * r / R, V * (R / np.maximum(r, 1e-6)) ** 0.5)
    return v.max(axis=1), (v >= 17).sum(axis=1) * 0.5

coast = adm3_utm.dissolve().boundary.iloc[0]
local_rows = []
for key in MODEL_KEYS:
    ev = EV.loc[key]
    sids = model_rows.loc[model_rows.event == key, "shapeID"]
    a = anchor[anchor.shapeID.isin(sids)]
    all_ = anchor_ll[anchor_ll.shapeID.isin(sids)]
    wmax, gale = wind_features(tracks[key], ev.landfall, all_.geometry.y.values, all_.geometry.x.values)
    area = adm3_utm.set_index("shapeID").loc[a.shapeID].area.values / 1e6
    local_rows.append(pd.DataFrame({"event": key, "shapeID": a.shapeID.values, "wind_ms": wmax.round(1),
                                    "gale_hours": gale, "dist_coast_km": (a.distance(coast) / 1000).round(2).values,
                                    "area_km2": area.round(2)}))
local = pd.concat(local_rows, ignore_index=True)
local.groupby("event")[["wind_ms", "gale_hours", "dist_coast_km"]].describe().T.round(1)

In [ ]:
# 8. Earth Engine feature export (one task per typhoon; existing files are skipped)
import geemap
EMB_BANDS = [f"A{i:02d}" for i in range(64)]

pop = (ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP").filterDate("2020-01-01", "2021-01-01").first()
         .select("population_count").multiply(100).rename("pop_density"))          # people per 100 m cell × 100 = people/km²
built = (ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S").filterDate("2020-01-01", "2021-01-01").first()
           .select("built_surface").divide(10000).rename("built_frac"))
elev = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("elev_m")

feat_tasks = {}
for key in MODEL_KEYS:
    if glob.glob(f"{DRIVE_DIR}/features_{key}*.csv"):
        print(key, "already exported, skipped")
        continue
    ev = EV.loc[key]
    t_utc = ev.landfall - pd.Timedelta(hours=8)
    rain = (ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
              .filterDate((t_utc - pd.Timedelta(days=1)).strftime("%Y-%m-%dT%H:%M:%S"),
                          (t_utc + pd.Timedelta(days=2)).strftime("%Y-%m-%dT%H:%M:%S"))
              .select("precipitation").sum().multiply(0.5).rename("rain_mm"))     # half-hourly mm/hr summed → mm
    y = ev.landfall.year - 1
    emb = (ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
             .filterDate(f"{y}-01-01", f"{y + 1}-01-01").mosaic().select(EMB_BANDS)
             .updateMask(built.gte(BUILT_MIN_FRAC)))
    stack = ee.Image.cat([pop, built, elev, rain, emb])
    sids = model_rows.loc[model_rows.event == key, "shapeID"]
    fc = geemap.gdf_to_ee(adm3[adm3.shapeID.isin(sids)][["shapeID", "geometry"]])
    table = stack.reduceRegions(fc, ee.Reducer.mean(), scale=100, tileScale=4)
    task = ee.batch.Export.table.toDrive(collection=table, description=f"features_{key}", folder=DRIVE_FOLDER,
                                         fileNamePrefix=f"features_{key}", fileFormat="CSV",
                                         selectors=["shapeID", "pop_density", "built_frac", "elev_m", "rain_mm"] + EMB_BANDS)
    task.start()
    feat_tasks[key] = task
    print(f"{key}: {len(sids)} municipalities, export started (embedding year {y})")

while feat_tasks:
    states = {k: t.status()["state"] for k, t in feat_tasks.items()}
    print(time.strftime("%H:%M:%S"), states)
    if all(s in ("COMPLETED", "FAILED", "CANCELLED") for s in states.values()):
        for k, t in feat_tasks.items():
            if t.status()["state"] == "FAILED":
                print(k, "FAILED:", t.status().get("error_message"))
        break
    time.sleep(60)

In [ ]:
# 9. Assemble the modelling table
ee_rows = []
for key in MODEL_KEYS:
    files = sorted(glob.glob(f"{DRIVE_DIR}/features_{key}*.csv"))
    if not files:
        print(f"{key}: no feature file yet (Drive may lag; rerun this cell shortly)")
        continue
    ee_rows.append(pd.concat([pd.read_csv(f) for f in files]).assign(event=key))
eef = pd.concat(ee_rows, ignore_index=True)

df = (model_rows[["event", "shapeID", "muni", "province", "dist_km", "eligible_px", "duration_days", "duration_censored"]]
        .merge(local, on=["event", "shapeID"], how="left")
        .merge(eef, on=["event", "shapeID"], how="left")
        .merge(pre_ntl, on=["event", "shapeID"], how="left"))

# Optional: Relative Wealth Index (RWI)
rwi_files = glob.glob(f"{DRIVE_DIR}/*relative_wealth*.csv")
if rwi_files:
    rwi = pd.read_csv(rwi_files[0])
    pts = gpd.GeoDataFrame(rwi, geometry=gpd.points_from_xy(rwi.longitude, rwi.latitude), crs=4326)
    m = gpd.sjoin(pts, adm3, predicate="within").groupby("shapeID").rwi.mean().rename("rwi")
    df = df.merge(m, on="shapeID", how="left")
    print(f"RWI: {df.rwi.notna().mean():.0%} of municipalities matched")
else:
    df["rwi"] = np.nan
    print("No RWI file; Part D will skip the RWI breakdown")

# Target: outage > 7 days. Rows censored at ≤ 7 days are undecidable and dropped
cens = df.duration_censored.astype(str).str.lower() == "true"
df = df[~(cens & (df.duration_days <= LONG_OUTAGE_DAYS))].copy()
df["y"] = (df.duration_days > LONG_OUTAGE_DAYS).astype(int)
df["log_pop_density"] = np.log1p(df.pop_density)
df["log_pre_ntl"] = np.log(df.pre_ntl.clip(lower=0.1))
df["log_area_km2"] = np.log(df.area_km2)
df = df.merge(anchor[["shapeID"]].assign(x_utm=anchor.geometry.x, y_utm=anchor.geometry.y), on="shapeID", how="left")

HAZARD = ["wind_ms", "gale_hours", "dist_km", "rain_mm"]
EXPOSURE = ["log_pop_density", "built_frac", "log_pre_ntl", "elev_m", "dist_coast_km", "log_area_km2"]
EMB = [c for c in EMB_BANDS if c in df.columns]
FEATURE_SETS = {"Hazard": HAZARD, "Hazard + exposure": HAZARD + EXPOSURE,
                "Hazard + exposure + embeddings": HAZARD + EXPOSURE + EMB}

df.to_csv(f"{DRIVE_DIR}/step3_features.csv", index=False, encoding="utf-8-sig")
print(df.groupby("event").agg(n=("y", "size"), long_outage_share=("y", "mean")).round(2))
print("Missing share:", df[HAZARD + EXPOSURE].isna().mean().round(2).to_dict(),
      "| embeddings:", round(df[EMB].isna().all(axis=1).mean(), 2) if EMB else "n/a")

## C. Models and three validation schemes

- **Random split** (5 folds × 3 repeats): training and test can come from the same typhoon and neighbouring municipalities; the most optimistic
- **Spatial blocks** (by province, 5 folds): no province appears in both training and test
- **Leave one typhoon out** (4 folds): train on 3 typhoons, predict the 4th; closest to real use

The metric is AUC (0.5 = random guessing, 1 = perfect separation). **The AUC gap between the random split and leave-one-typhoon-out is the "transfer loss" of RQ2**. If the embeddings help, they should narrow that gap.

Within each training fold, the 64 embedding dimensions are first reduced to 10 principal components. This avoids leakage and avoids overfitting 70-odd features on about 550 samples.

In [ ]:
# 10. Train and evaluate
import warnings
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
warnings.filterwarnings("ignore", category=UserWarning)

N_PCA = 10
MODELS = ["Logistic regression", "Gradient boosting"]
SCHEMES = ["Random split", "Spatial blocks (province)", "Leave one typhoon out"]

def make_model(kind, cols):
    emb = [c for c in cols if c in EMB]
    other = [c for c in cols if c not in EMB]
    parts = [("other", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), other)]
    if emb:
        parts.append(("emb", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                                       ("pca", PCA(n_components=N_PCA, random_state=0))]), emb))
    clf = (LogisticRegression(C=1.0, max_iter=5000) if kind == MODELS[0] else
           HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                          l2_regularization=1.0, random_state=0))
    return Pipeline([("pre", ColumnTransformer(parts)), ("clf", clf)])

def split_iter(scheme, data, seed):
    y = data.y.values
    if scheme == SCHEMES[0]:
        return StratifiedKFold(5, shuffle=True, random_state=seed).split(data, y)
    if scheme == SCHEMES[1]:
        return GroupKFold(5).split(data, y, groups=data.province.fillna("unknown"))
    return LeaveOneGroupOut().split(data, y, groups=data.event)

fold_rows, oof = [], {}
data = df.reset_index(drop=True)
for scheme in SCHEMES:
    seeds = [0, 1, 2] if scheme == SCHEMES[0] else [0]
    for kind in MODELS:
        for fs_name, cols in FEATURE_SETS.items():
            if not cols:
                continue
            for seed in seeds:
                pred = np.full(len(data), np.nan)
                for fold, (tr, te) in enumerate(split_iter(scheme, data, seed)):
                    m = make_model(kind, cols).fit(data.loc[tr, cols], data.y[tr])
                    p = m.predict_proba(data.loc[te, cols])[:, 1]
                    pred[te] = p
                    yt = data.y[te]
                    fold_rows.append({"scheme": scheme, "model": kind, "features": fs_name, "seed": seed, "fold": fold,
                                      "held_out": data.event[te].iloc[0] if scheme == SCHEMES[2] else "",
                                      "n_test": len(te),
                                      "auc": roc_auc_score(yt, p) if yt.nunique() == 2 else np.nan,
                                      "ap": average_precision_score(yt, p) if yt.nunique() == 2 else np.nan,
                                      "brier": brier_score_loss(yt, p)})
                if seed == 0:
                    oof[(scheme, kind, fs_name)] = pred
    print(scheme, "done")

folds = pd.DataFrame(fold_rows)
res = (folds.groupby(["scheme", "model", "features"], sort=False)
            .agg(auc_mean=("auc", "mean"), auc_sd=("auc", "std"), ap_mean=("ap", "mean"), brier=("brier", "mean"))
            .reset_index())
res["pooled_auc"] = [roc_auc_score(data.y, oof[(r.scheme, r.model, r.features)]) for r in res.itertuples()]
res.round(3)

In [ ]:
# 11. Figure: AUC under the three validation schemes
COLORS = dict(zip(FEATURE_SETS, [C1, C2, C3]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=True, constrained_layout=True)
for ax, kind in zip(axes, MODELS):
    r = res[res.model == kind]
    for i, fs in enumerate(FEATURE_SETS):
        s = r[r.features == fs].set_index("scheme").reindex(SCHEMES)
        x = np.arange(len(SCHEMES)) + (i - 1) * 0.18
        ax.errorbar(x, s.auc_mean, yerr=s.auc_sd, fmt="o", ms=8, color=COLORS[fs], ecolor=COLORS[fs],
                    elinewidth=1.5, capsize=0, label=fs)
    ax.axhline(0.5, color="#a3a29d", lw=1, ls="--")
    ax.text(len(SCHEMES) - 0.5, 0.51, "chance", color=INK2, fontsize=8, ha="right", va="bottom")
    ax.set_xticks(range(len(SCHEMES)), ["Random\nsplit", "Spatial blocks\n(province)", "Leave one\ntyphoon out"])
    ax.set_ylim(0.4, 1.0)
    ax.set_title(kind, loc="left", fontsize=11)
    ax.grid(axis="y", color=GRID, lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_ylabel("AUC (mean ± SD across folds)")
axes[1].legend(frameon=False, fontsize=9, loc="lower left")
fig.suptitle(f"Predicting outages longer than {LONG_OUTAGE_DAYS} days: does performance hold as the test gets harder?",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step3_auc_by_scheme.png", dpi=200, bbox_inches="tight")
plt.show()

gap = res.pivot_table(index=["model", "features"], columns="scheme", values="auc_mean", sort=False)[SCHEMES]
gap["transfer loss (random − leave-one-typhoon-out)"] = gap[SCHEMES[0]] - gap[SCHEMES[2]]
gap.round(3)

In [ ]:
# 12. AUC for each held-out typhoon
loeo = folds[folds.scheme == SCHEMES[2]].pivot_table(index=["model", "features"], columns="held_out",
                                                       values="auc", sort=False)
display(loeo.round(3))
print("Share with outage > 7 days per typhoon:", data.groupby("event").y.mean().round(2).to_dict())

## D. Diagnostics

**Spatial autocorrelation of residuals (Moran's I)**: using the leave-one-typhoon-out predictions, compute Moran's I of the residuals (observed − predicted probability) within each typhoon, with the 6 nearest neighbours of each municipality.
- I near 0: the model has captured the spatial structure
- I clearly positive: neighbouring municipalities err in the same direction, so a spatially structured factor is missing (for example grid zones, or the capacity of the electric cooperative). This is the "spatial explicitness" test proposed by Zhou et al. (2026)

**Performance by group (a preview of RQ3)**: compare model performance across terciles of pre-event light (and RWI, if available), especially the **miss rate**: the share of municipalities that truly had an outage longer than 7 days but that the model did not flag.

In [ ]:
# 13. Moran's I of residuals (within each typhoon, kNN = 6, 999 permutations)
from sklearn.neighbors import NearestNeighbors

def morans_i(z, xy, k=6, n_perm=999, seed=0):
    idx = NearestNeighbors(n_neighbors=k + 1).fit(xy).kneighbors(xy, return_distance=False)[:, 1:]
    z = z - z.mean()
    stat = lambda v: (v * v[idx].mean(axis=1)).sum() / (v ** 2).sum()
    obs = stat(z)
    rng = np.random.default_rng(seed)
    perm = np.array([stat(rng.permutation(z)) for _ in range(n_perm)])
    return obs, (1 + (perm >= obs).sum()) / (n_perm + 1)

mor_rows = []
for kind in MODELS:
    for fs in FEATURE_SETS:
        p = oof[(SCHEMES[2], kind, fs)]
        for e, g in data.assign(p=p).groupby("event"):
            g = g.dropna(subset=["x_utm", "y_utm"])
            if len(g) < 20:
                continue
            I, pv = morans_i((g.y - g.p).values, g[["x_utm", "y_utm"]].values)
            mor_rows.append({"model": kind, "features": fs, "event": e, "n": len(g),
                             "Moran's I": round(I, 3), "p-value": round(pv, 3)})
MOR = pd.DataFrame(mor_rows)
MOR.pivot_table(index=["model", "features"], columns="event", values="Moran's I", sort=False)

In [ ]:
# 14. Performance by group: terciles of pre-event light (and RWI), using leave-one-typhoon-out predictions
BEST = ("Gradient boosting", "Hazard + exposure + embeddings")   # change to the best combination from cell 11 if needed
d = data.assign(p=oof[(SCHEMES[2],) + BEST])

def group_table(col, labels_):
    g = d.dropna(subset=[col]).copy()
    g["group"] = pd.qcut(g[col], 3, labels=labels_)
    return g.groupby("group", observed=True).apply(lambda x: pd.Series({
        "municipalities": len(x),
        "share > 7 days": x.y.mean(),
        "mean predicted": x.p.mean(),
        "AUC": roc_auc_score(x.y, x.p) if x.y.nunique() == 2 else np.nan,
        "miss rate (y=1, p<0.5)": ((x.p < 0.5) & (x.y == 1)).sum() / max((x.y == 1).sum(), 1),
    }), include_groups=False)

print(f"{BEST[0]} · {BEST[1]}")
print("\nBy pre-event light:")
display(group_table("log_pre_ntl", ["Dim", "Middle", "Bright"]).round(3))
if d.rwi.notna().sum() > 30:
    print("\nBy Relative Wealth Index:")
    display(group_table("rwi", ["Poorest third", "Middle", "Richest third"]).round(3))

In [ ]:
# 15. Save and download
res.to_csv("figures/step3_results.csv", index=False, encoding="utf-8-sig")
folds.to_csv("figures/step3_folds.csv", index=False, encoding="utf-8-sig")
PT.to_csv("figures/step3_placebo.csv", encoding="utf-8-sig")
MOR.to_csv("figures/step3_morans_i.csv", index=False, encoding="utf-8-sig")
for f in ["step3_results.csv", "step3_placebo.csv", "step3_morans_i.csv"]:
    os.system(f'cp "figures/{f}" "{DRIVE_DIR}/{f}"')
from google.colab import files
for f in ["step3_results.csv", "step3_folds.csv", "step3_placebo.csv", "step3_morans_i.csv",
          "step3_placebo.png", "step3_auc_by_scheme.png"]:
    files.download(f"figures/{f}")